In [4]:
from os import getcwd

import polars as pl
import pandas as pd
import pandera.polars as pa

from pathlib import Path

from datetime import datetime

In [17]:
a.combined_df.shape

(144788, 6)

In [16]:
from traffic_data import DataImport

a = DataImport()
a.save_combined()

In [15]:
time_type = ["year", "month", "week", "day"]

a.combined_df.with_columns([
    getattr(pl.col("Time").dt, i)().alias(i) for i in time_type
])

Time of occurrence,Type of treatment,Location of accident,CoordinateX,CoordinateY,Time,year,month,week,day
str,i64,str,f64,f64,datetime[μs],i32,i8,i8,i8
"""""2019/1/2-08:37""""",2,"""大同區重慶北路3段137巷與民族西路182巷口""",121.514854,25.068009,2019-01-02 08:37:00,2019,1,1,2
"""""2019/1/9-10:59""""",2,"""大同區重慶北路1段與天水路口""",121.514213,25.053566,2019-01-09 10:59:00,2019,1,2,9
"""""2019/1/18-09:54""""",2,"""大同區民權西路與蘭州街口""",121.514705,25.062889,2019-01-18 09:54:00,2019,1,3,18
"""""2019/1/23-08:31""""",2,"""大同區承德路1段與長安西路口""",121.5171,25.050867,2019-01-23 08:31:00,2019,1,4,23
"""""2019/1/28-18:45""""",2,"""大同區延平北路2段61巷與延平北路2段口口""",121.512052,25.055509,2019-01-28 18:45:00,2019,1,5,28
…,…,…,…,…,…,…,…,…,…
"""2024/12/31 15:10""",2,"""內湖區行善路70號附近側附近""",121.576831,25.057858,2024-12-31 15:10:00,2024,12,1,31
"""2024/12/31 15:33""",2,"""內湖區瑞湖街與陽光街321巷口""",121.577269,25.071146,2024-12-31 15:33:00,2024,12,1,31
"""2024/12/31 18:30""",2,"""內湖區民權大橋東向西上橋處""",121.577348,25.066412,2024-12-31 18:30:00,2024,12,1,31


In [ ]:
    # Ordered list of address components and their regex patterns
    structures = [
        ("district", r".+?區"),                      # Matches district ending with '區'
        ("road", r".+?(路|街|大道|橋|圓環|廣場)"),   # Matches road names with common suffixes
        ("section", r"\d+段"),                      # Matches section numbers, e.g., '3段'
        ("lane", r"\d+巷"),                         # Matches lane numbers, e.g., '137巷'
        ("alley", r"\d+弄|\d+衖"),                  # Matches alleys, e.g., '20弄' or '20衖'
        ("number", r"\d+號"),                        # Matches building numbers, e.g., '5號'
        ("floor", r"\d+樓"),                         # Matches floor numbers, e.g., '3樓'
        ("room", r"\d+室"),                          # Matches room numbers, e.g., '101室'
    ]

In [115]:
import polars as pl

# 1. List of possible formats
formats = [
    "%Y/%m/%d %H:%M",
    "%Y-%m-%d %H:%M",
    "%Y/%m/%d %H:%M:%S",
    "%Y-%m-%d %H:%M:%S"
]

# 2. Clean the strings
df = a.combined_df.with_columns(
    pl.col("Time of occurrence")
    .cast(pl.Utf8)
    .str.strip_chars('" ')
    .str.replace("-", "/", literal=False)
    .alias("Time_clean")
)

# 3. Try multiple formats
parsed_exprs = [pl.col("Time_clean").str.strptime(pl.Datetime, fmt, strict=False) for fmt in formats]

df = df.with_columns(
    pl.coalesce(parsed_exprs).alias("Time_parsed")  # takes first successful parse
)

# 4. Extract components
df = df.with_columns([
    pl.col("Time_parsed").dt.year().alias("year"),
    pl.col("Time_parsed").dt.month().alias("month"),
    pl.col("Time_parsed").dt.week().alias("week"),
    pl.col("Time_parsed").dt.day().alias("day"),
    pl.col("Time_parsed").dt.hour().alias("hour"),
])

a.combined_df = df


In [124]:
print(a.combined_df.describe())

shape: (9, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ Time of   ┆ Type of   ┆ Location  ┆ … ┆ day       ┆ hour      ┆ Time_clea ┆ Time_par │
│ ---       ┆ occurrenc ┆ treatment ┆ of        ┆   ┆ ---       ┆ ---       ┆ n         ┆ sed      │
│ str       ┆ e         ┆ ---       ┆ accident  ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---      │
│           ┆ ---       ┆ f64       ┆ ---       ┆   ┆           ┆           ┆ str       ┆ str      │
│           ┆ str       ┆           ┆ str       ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 144788    ┆ 144788.0  ┆ 144788    ┆ … ┆ 121870.0  ┆ 121870.0  ┆ 144788    ┆ 121870   │
│ null_coun ┆ 0         ┆ 0.0       ┆ 0         ┆ … ┆ 22918.0   ┆ 22918.0   ┆ 0         ┆ 22918    │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           

In [14]:

a.combined_df['datetime'] = a.combined_df['Time of occurrence'].apply(
	lambda x: datetime.strptime(x.strip('"').replace("-", " "), given_format) if pd.notnull(x)
	else pd.NaT
)

AttributeError: 'Series' object has no attribute 'apply'

In [ ]:
a.    def add_datetime(self, data):
        given_format = "%Y/%m/%d %H:%M"

        data['datetime'] = data['Time of occurrence'].apply(
            lambda x: datetime.strptime(x.strip('"').replace("-", " "), given_format) if pd.notnull(x) else pd.NaT
        )

        data['year'] = data['datetime'].dt.year
        data['month'] = data['datetime'].dt.month
        data['week'] = data['datetime'].dt.isocalendar().week
        data['day'] = data['datetime'].dt.day
        data['hour'] = data['datetime'].dt.hour
        return data